In [ ]:
import os
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt 
import numpy as np 
import math
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, callbacks
from sklearn.utils import class_weight

In [ ]:
train_dir='/kaggle/input/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train'
valid_dir='/kaggle/input/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/valid'
test_dir='/kaggle/input/new-plant-diseases-dataset/test'

In [ ]:
nums = {}
for dir_name in sorted(os.listdir(train_dir)):  # Sort here
    class_dir = os.path.join(train_dir, dir_name)
    if os.path.isdir(class_dir):
        nums[dir_name] = len(os.listdir(class_dir))

n_train = sum(nums.values())
print(f"There are {n_train} images for training")

img_per_class = pd.DataFrame.from_dict(nums, orient='index', columns=["no. of images"])
img_per_class.index.name = "class name"
img_per_class = img_per_class.sort_index()

img_per_class

In [ ]:
import random
from PIL import Image

def show_one_image_per_class_from_folders(train_dir, image_size=(224, 224)):
    class_names = sorted(os.listdir(train_dir))
    cols = 5
    rows = math.ceil(len(class_names) / cols)

    plt.figure(figsize=(cols * 3, rows * 3))

    for i, class_name in enumerate(class_names):
        class_path = os.path.join(train_dir, class_name)
        if not os.path.isdir(class_path):
            continue

        image_files = [f for f in os.listdir(class_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        if not image_files:
            continue

        chosen_image = random.choice(image_files)
        image_path = os.path.join(class_path, chosen_image)

        image = Image.open(image_path).convert("RGB").resize(image_size)

        ax = plt.subplot(rows, cols, i + 1)
        plt.imshow(image)
        plt.title(class_name, fontsize=10)
        plt.axis("off")

    for j in range(len(class_names), rows * cols):
        ax = plt.subplot(rows, cols, j + 1)
        ax.axis("off")

    plt.tight_layout()
    plt.show()


show_one_image_per_class_from_folders(train_dir)

In [ ]:
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset='training',
    batch_size=32,
    image_size=(224, 224), 
    seed=123,
    shuffle=True,
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    val_dir,
    validation_split=0.2,
    subset='validation',
    batch_size=32,
    image_size=(224, 224),
    seed=42,
    shuffle=True,
)

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(height_factor=(-0.2, 0.2), width_factor=(-0.2, 0.2)),
    tf.keras.layers.RandomContrast(factor=0.2)
])

In [ ]:
class_names = train_ds.class_names
num_classes = len(class_names)

print("Number of Classes:", num_classes)
print("Class Names (ordered by assigned integer label):", class_names)

print("\nCollecting all training labels for class weight calculation...")

all_training_labels = []
for images, labels in train_ds.unbatch():
    all_training_labels.append(labels.numpy())

all_training_labels_flat = np.array(all_training_labels).flatten()

unique_classes = np.arange(num_classes)
index_to_class_name = dict(zip(unique_classes, class_names))

sklearn_class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=unique_classes,
    y=all_training_labels_flat
)

class_weights_dict = dict(enumerate(sklearn_class_weights))

In [ ]:
base_model = tf.keras.applications.ResNet50(include_top=False,
                                            weights='/kaggle/input/resnet50/tensorflow2/imagenet/1/resnet50_weights_tf_dim_ordering_tf_kernels_notop.h5',
                                            input_shape=(224, 224, 3))

base_model.trainable = False

for layer in base_model.layers[-10:]:
    layer.trainable = True

inputs = tf.keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = tf.keras.applications.resnet.preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(512, activation='relu')(x)
x = tf.keras.layers.Dropout(0.5)(x)
outputs = tf.keras.layers.Dense(num_classes)(x)

model = tf.keras.Model(inputs=inputs, outputs=outputs)

optimizer = tf.keras.optimizers.Adam(learning_rate=1e-5)
model.compile(
    optimizer=optimizer,
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)


In [ ]:
early_stopping = callbacks.EarlyStopping(patience=3)

history = model.fit(
    train_ds,
    epochs=10,
    validation_data=val_ds,
    callbacks=[early_stopping],  
    class_weight=class_weights_dict
)

model.save_weights('/kaggle/working/resnet50_plantvillage.weights.h5')

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(len(acc))

plt.figure(figsize=(15,4))
plt.subplot(1,2,1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.legend()

plt.subplot(1,2,2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.show()

In [ ]:

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

y_true = []
y_pred = []

for images, labels in val_ds:
    predictions = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(predictions, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
cm = confusion_matrix(y_true, y_pred)

print("📊 Validation Evaluation Metrics:")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")
print()

print("🧾 Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix (Validation Set)')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.show()

In [ ]:
def load_and_preprocess_image(img_path):
    img = tf.keras.utils.load_img(img_path, target_size=(224, 224))
    img_array = tf.keras.utils.img_to_array(img)
    img_array = tf.expand_dims(img_array, axis=0)
    img_array = tf.keras.applications.resnet.preprocess_input(img_array)
    return img_array

results = []
for filename in sorted(os.listdir(test_dir + "/test")):
    if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
        img_path = os.path.join(test_dir + "/test", filename)
        img_array = load_and_preprocess_image(img_path)

        logits = model.predict(img_array)
        probs = tf.nn.softmax(logits, axis=-1).numpy()
        predicted_index = tf.argmax(probs, axis=-1).numpy()[0]
        confidence = probs[0][predicted_index]
        predicted_label = index_to_class_name[predicted_index]

        results.append({
            "filename": filename,
            "img_path": img_path,
            "label": predicted_label,
            "confidence": confidence
        })

plt.figure(figsize=(20, 20))
for i, result in enumerate(results[:25]):
    plt.subplot(5, 5, i + 1)
    img = tf.keras.utils.load_img(result["img_path"], target_size=(224, 224))
    plt.imshow(img)
    plt.title(f"Actual: {result['filename']}\nPredicted: {result['label']} ({result['confidence'] * 100:.1f}%)", fontsize=11)
    plt.axis("off")
plt.tight_layout()
plt.show()